In [ ]:
def simulate_auction(bids_existing, asks_existing, my_bid_price, my_bid_qty, buyback_price, fee_per_unit):
    """
    Simulate a single-price auction.
    
    bids_existing: list of (price, volume) - existing bids (other participants)
    asks_existing: list of (price, volume) - existing asks (other participants)
    my_bid_price: my bid price
    my_bid_qty: my bid quantity
    buyback_price: guaranteed buyback price after auction
    fee_per_unit: fee per unit traded on buyback
    
    Returns: (clearing_price, my_filled_qty, pnl)
    """
    
    # Add my bid to the book (I'm last in time priority)
    all_bids = [(p, v, 0) for p, v in bids_existing] + [(my_bid_price, my_bid_qty, 1)]  # 1 = me (last priority)
    all_asks = [(p, v) for p, v in asks_existing]
    
    if not all_asks:
        return None, 0, 0
    
    # Collect all candidate clearing prices
    candidate_prices = set()
    for p, v, _ in all_bids:
        candidate_prices.add(p)
    for p, v in all_asks:
        candidate_prices.add(p)
    
    candidate_prices = sorted(candidate_prices)
    
    best_cp = None
    best_volume = -1
    
    for cp in candidate_prices:
        total_bid_vol = sum(v for p, v, _ in all_bids if p >= cp)
        total_ask_vol = sum(v for p, v in all_asks if p <= cp)
        traded_vol = min(total_bid_vol, total_ask_vol)
        
        if traded_vol > best_volume or (traded_vol == best_volume and cp > best_cp):
            best_volume = traded_vol
            best_cp = cp
    
    if best_cp is None or best_volume == 0:
        return None, 0, 0
    
    clearing_price = best_cp
    
    # Determine how much I get filled
    # Ask side: total ask volume available at clearing price
    total_ask_vol = sum(v for p, v in all_asks if p <= clearing_price)
    
    # Bid side: sort by price desc (price priority), then time priority (me last)
    # All bids with price >= clearing_price participate
    eligible_bids = [(p, v, who) for p, v, who in all_bids if p >= clearing_price]
    # Sort: higher price first, then who=0 (existing) before who=1 (me)
    eligible_bids.sort(key=lambda x: (-x[0], x[2]))
    
    remaining_ask = total_ask_vol
    my_filled = 0
    
    for p, v, who in eligible_bids:
        fill = min(v, remaining_ask)
        remaining_ask -= fill
        if who == 1:
            my_filled = fill
    
    # PnL: buy at clearing_price, sell at buyback_price, pay fee
    pnl = my_filled * (buyback_price - clearing_price - fee_per_unit)
    
    return clearing_price, my_filled, pnl


def find_best_order(bids_existing, asks_existing, buyback_price, fee_per_unit, price_range, volume_range, top_n=10):
    """
    Brute force search over price and volume to find best PnL.
    
    price_range: (min_price, max_price) — integers
    volume_range: (min_vol, max_vol) — integers
    top_n: number of top results to display
    """
    results = []
    
    for price in range(price_range[0], price_range[1] + 1):
        for vol in range(volume_range[0], volume_range[1] + 1):
            cp, filled, pnl = simulate_auction(bids_existing, asks_existing, price, vol, buyback_price, fee_per_unit)
            if pnl > 0:
                results.append((pnl, price, vol, cp, filled))
    
    # Sort by PnL descending
    results.sort(key=lambda x: -x[0])
    
    print(f"Top {top_n} results:")
    print(f"{'Rank':<6} {'Bid Price':<12} {'Bid Vol':<12} {'Clearing P':<12} {'Filled':<12} {'PnL':<12}")
    print("-" * 66)
    for i, (pnl, price, vol, cp, filled) in enumerate(results[:top_n]):
        print(f"{i+1:<6} {price:<12} {vol:<12} {cp:<12} {filled:<12} {pnl:<12.2f}")
    
    if results:
        best_pnl, best_price, best_vol, best_cp, best_filled = results[0]
        return best_price, best_vol, best_pnl
    else:
        print("No profitable orders found.")
        return None, None, 0

In [21]:
print("=== DRYLAND FLAX ===")
flax_bids = [
    (30, 30000),
    (29, 5000),
    (28, 12000),
    (27, 28000),
]

flax_asks = [
    (28, 40000),
    (31, 20000),
    (32, 20000),
    (33, 30000),
]
find_best_order(flax_bids, flax_asks, buyback_price=30, fee_per_unit=0, price_range=(0, 50), volume_range=(0, 30000))



=== DRYLAND FLAX ===
Top 10 results:
Rank   Bid Price    Bid Vol      Clearing P   Filled       PnL         
------------------------------------------------------------------
1      30           9999         29           9999         9999.00     
2      31           9999         29           9999         9999.00     
3      32           9999         29           9999         9999.00     
4      33           9999         29           9999         9999.00     
5      34           9999         29           9999         9999.00     
6      35           9999         29           9999         9999.00     
7      36           9999         29           9999         9999.00     
8      37           9999         29           9999         9999.00     
9      38           9999         29           9999         9999.00     
10     39           9999         29           9999         9999.00     


(30, 9999, 9999)

In [22]:
print("=== EMBER MUSHROOM ===")
mushroom_bids = [
    (20, 43000),
    (19, 17000),
    (18, 6000),
    (17, 5000),
    (16, 10000),
    (15, 5000),
    (14, 10000),
    (13, 7000),
]

mushroom_asks = [
    (12, 20000),
    (13, 25000),
    (14, 35000),
    (15, 6000),
    (16, 5000),
    (17, 0),
    (18, 10000),
    (19, 12000),
]
find_best_order(mushroom_bids, mushroom_asks, buyback_price=20, fee_per_unit=0.10, price_range=(0, 30), volume_range=(0, 40000))

=== EMBER MUSHROOM ===
Top 10 results:
Rank   Bid Price    Bid Vol      Clearing P   Filled       PnL         
------------------------------------------------------------------
1      17           19999        16           19999        77996.10    
2      18           19999        16           19999        77996.10    
3      19           19999        16           19999        77996.10    
4      20           19999        16           19999        77996.10    
5      21           19999        16           19999        77996.10    
6      22           19999        16           19999        77996.10    
7      23           19999        16           19999        77996.10    
8      24           19999        16           19999        77996.10    
9      25           19999        16           19999        77996.10    
10     26           19999        16           19999        77996.10    


(17, 19999, 77996.09999999999)